# 04 — A1 Phase-Aware U-Net Training

This is the first controlled ablation after A0.

**A0:** magnitude-only U-Net + noisy phase  
**A1:** same U-Net backbone + magnitude mask + learned phase correction

The following are intentionally held fixed relative to A0:

- VoiceBank+DEMAND train/validation manifests
- 16 kHz sample rate
- STFT: 512 FFT, 400-sample Hann window, 100-sample hop
- 2.56 s training segments
- base channels = 16, depth = 4
- AdamW, learning rate = 2e-4, weight decay = 1e-5
- batch size = 8
- early stopping patience = 12
- random seed = 42

**Project root:** `D:\PAPERS\SPEECH\low_snr_speech_enhancement`

The only substantive model change is explicit phase estimation. The A1 loss therefore adds
phase-aware supervision to the same magnitude-reconstruction objective used by A0.

In [ ]:
from pathlib import Path
import os, random, json, math
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement")
TRAIN_MANIFEST = PROJECT_ROOT / "manifests" / "voicebank" / "train.csv"
VAL_MANIFEST   = PROJECT_ROOT / "manifests" / "voicebank" / "val.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "a1_phaseaware"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_MANIFEST.exists(), f"Missing train manifest: {TRAIN_MANIFEST}"
assert VAL_MANIFEST.exists(), f"Missing validation manifest: {VAL_MANIFEST}"

SEED = 42
SR = 16000
N_FFT = 512
WIN_LENGTH = 400
HOP_LENGTH = 100
SEGMENT_SECONDS = 2.56

BATCH_SIZE = 8
EPOCHS = 100
LR = 2e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 12
GRAD_CLIP = 5.0

# Phase-aware loss weights fixed for A1.
LAMBDA_COMPLEX = 0.10
LAMBDA_PHASE = 0.05

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Initial allocated GPU memory (MB):",
          torch.cuda.memory_allocated() / (1024**2))
print("Train manifest:", TRAIN_MANIFEST)
print("Validation manifest:", VAL_MANIFEST)

## Dataset

In [ ]:
def read_audio(path, target_sr=16000):
    wav, sr = sf.read(path, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)

    if sr != target_sr:
        wav = librosa.resample(
            wav,
            orig_sr=sr,
            target_sr=target_sr,
            res_type="kaiser_best"
        ).astype(np.float32)

    return wav

def pad_or_crop_pair(clean, noisy, target_len, training):
    n = min(len(clean), len(noisy))
    clean = clean[:n]
    noisy = noisy[:n]

    if n >= target_len:
        if training:
            start = random.randint(0, n - target_len)
        else:
            start = 0
        return (
            clean[start:start + target_len],
            noisy[start:start + target_len]
        )

    pad = target_len - n
    return (
        np.pad(clean, (0, pad)),
        np.pad(noisy, (0, pad))
    )

class PairedSpeechDataset(Dataset):
    def __init__(self, manifest, training=False):
        self.df = pd.read_csv(manifest)
        self.training = training
        self.target_len = int(SR * SEGMENT_SECONDS)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        clean = read_audio(row.clean_path, SR)
        noisy = read_audio(row.noisy_path, SR)

        clean, noisy = pad_or_crop_pair(
            clean,
            noisy,
            self.target_len,
            self.training
        )

        peak = max(
            float(np.max(np.abs(clean))),
            float(np.max(np.abs(noisy))),
            1e-8
        )

        if peak > 1.0:
            clean = clean / peak
            noisy = noisy / peak

        return torch.from_numpy(clean), torch.from_numpy(noisy)

train_ds = PairedSpeechDataset(TRAIN_MANIFEST, training=True)
val_ds = PairedSpeechDataset(VAL_MANIFEST, training=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

print("Train utterances:", len(train_ds))
print("Validation utterances:", len(val_ds))

## STFT utilities

In [ ]:
def stft_complex(waveform):
    window = torch.hann_window(
        WIN_LENGTH,
        device=waveform.device,
        dtype=waveform.dtype
    )
    return torch.stft(
        waveform,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        return_complex=True
    )

def istft_complex(spec, length):
    window = torch.hann_window(
        WIN_LENGTH,
        device=spec.device,
        dtype=spec.real.dtype
    )
    return torch.istft(
        spec,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        length=length
    )

## A1 model

The magnitude branch remains sigmoid-mask based as in A0.

The phase branch predicts a 2-D vector `(u, v)` for every time-frequency bin.
After normalization, the phase correction is:

`delta_phase = atan2(v, u)`

The phase-head bias is initialized to `(1, 0)`, corresponding to zero phase correction.
This gives the model a stable starting point equivalent to reusing noisy phase.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class PhaseAwareUNet(nn.Module):
    def __init__(self, base_channels=16, depth=4):
        super().__init__()

        channels = [base_channels * (2 ** i) for i in range(depth)]

        self.encoders = nn.ModuleList()
        in_ch = 1
        for ch in channels:
            self.encoders.append(ConvBlock(in_ch, ch))
            in_ch = ch

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(channels[-1], channels[-1] * 2)

        self.up_convs = nn.ModuleList()
        self.decoders = nn.ModuleList()

        dec_in = channels[-1] * 2
        for ch in reversed(channels):
            self.up_convs.append(
                nn.Conv2d(dec_in, ch, kernel_size=1)
            )
            self.decoders.append(
                ConvBlock(ch * 2, ch)
            )
            dec_in = ch

        self.mag_head = nn.Conv2d(
            channels[0], 1, kernel_size=1
        )

        # Two outputs: u and v for circular phase representation.
        self.phase_head = nn.Conv2d(
            channels[0], 2, kernel_size=1
        )

        # Stable initialization: delta phase starts at ~0 radians.
        nn.init.zeros_(self.phase_head.weight)
        with torch.no_grad():
            self.phase_head.bias[0] = 1.0
            self.phase_head.bias[1] = 0.0

    def forward(self, noisy_spec):
        noisy_mag = noisy_spec.abs()
        noisy_phase = torch.angle(noisy_spec)

        x = torch.log1p(noisy_mag).unsqueeze(1)

        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(
            self.up_convs,
            self.decoders,
            reversed(skips)
        ):
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        # Same bounded mask family as A0.
        mag_mask = torch.sigmoid(
            self.mag_head(x)
        ).squeeze(1)

        enhanced_mag = mag_mask * noisy_mag

        phase_uv = self.phase_head(x)
        u = phase_uv[:, 0]
        v = phase_uv[:, 1]

        norm = torch.sqrt(
            u.square() + v.square() + 1e-8
        )
        u = u / norm
        v = v / norm

        delta_phase = torch.atan2(v, u)
        enhanced_phase = noisy_phase + delta_phase

        enhanced_spec = torch.polar(
            enhanced_mag,
            enhanced_phase
        )

        return {
            "enhanced_spec": enhanced_spec,
            "enhanced_mag": enhanced_mag,
            "enhanced_phase": enhanced_phase,
            "mag_mask": mag_mask,
            "delta_phase": delta_phase
        }

model = PhaseAwareUNet(
    base_channels=16,
    depth=4
).to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Synthetic forward-pass check

In [ ]:
model.eval()

with torch.no_grad():
    test_len = int(SR * SEGMENT_SECONDS)
    clean_test = torch.randn(
        2, test_len, device=device
    ) * 0.03
    noisy_test = (
        clean_test +
        torch.randn_like(clean_test) * 0.02
    )

    noisy_spec_test = stft_complex(noisy_test)
    out_test = model(noisy_spec_test)

    reconstructed = istft_complex(
        out_test["enhanced_spec"],
        length=test_len
    )

assert out_test["enhanced_spec"].shape == noisy_spec_test.shape
assert reconstructed.shape == noisy_test.shape
assert torch.isfinite(reconstructed).all()
assert torch.isfinite(out_test["delta_phase"]).all()

print("Input waveform:", tuple(noisy_test.shape))
print("STFT:", tuple(noisy_spec_test.shape))
print("Enhanced spectrum:", tuple(out_test["enhanced_spec"].shape))
print("Reconstructed waveform:", tuple(reconstructed.shape))
print("Mean absolute initial phase correction:",
      float(out_test["delta_phase"].abs().mean().cpu()))
print("A1 synthetic forward test: PASS")

model.train()

## Phase-aware loss

`L_total = L_mag + 0.10 L_complex + 0.05 L_phase`

- `L_mag`: same log-magnitude L1 objective used by A0.
- `L_complex`: L1 error on real and imaginary components.
- `L_phase`: circular phase loss `1 - cos(error)`, weighted by clean-speech magnitude.

The circular form avoids discontinuity at `-π` / `+π`.

In [ ]:
def phase_aware_loss(
    outputs,
    clean_spec,
    lambda_complex=LAMBDA_COMPLEX,
    lambda_phase=LAMBDA_PHASE
):
    clean_mag = clean_spec.abs()
    clean_phase = torch.angle(clean_spec)

    enhanced_spec = outputs["enhanced_spec"]
    enhanced_mag = outputs["enhanced_mag"]
    enhanced_phase = outputs["enhanced_phase"]

    # A0-compatible magnitude loss.
    loss_mag = F.l1_loss(
        torch.log1p(enhanced_mag),
        torch.log1p(clean_mag)
    )

    # Complex-domain fidelity.
    loss_complex = 0.5 * (
        F.l1_loss(enhanced_spec.real, clean_spec.real) +
        F.l1_loss(enhanced_spec.imag, clean_spec.imag)
    )

    # Circular phase error.
    phase_error = 1.0 - torch.cos(
        enhanced_phase - clean_phase
    )

    # Speech-dominant TF bins receive more weight.
    weight = clean_mag / (
        clean_mag.mean(
            dim=(-2, -1),
            keepdim=True
        ) + 1e-8
    )
    weight = torch.clamp(weight, max=10.0)

    loss_phase = (
        phase_error * weight
    ).sum() / (
        weight.sum() + 1e-8
    )

    total = (
        loss_mag +
        lambda_complex * loss_complex +
        lambda_phase * loss_phase
    )

    return {
        "total": total,
        "mag": loss_mag,
        "complex": loss_complex,
        "phase": loss_phase
    }

## Validation function

In [ ]:
@torch.no_grad()
def validate():
    model.eval()

    totals = {
        "total": 0.0,
        "mag": 0.0,
        "complex": 0.0,
        "phase": 0.0
    }
    count = 0

    for clean, noisy in val_loader:
        clean = clean.to(
            device,
            non_blocking=True
        )
        noisy = noisy.to(
            device,
            non_blocking=True
        )

        clean_spec = stft_complex(clean)
        noisy_spec = stft_complex(noisy)

        outputs = model(noisy_spec)
        losses = phase_aware_loss(
            outputs,
            clean_spec
        )

        b = clean.size(0)
        for k in totals:
            totals[k] += (
                losses[k].item() * b
            )
        count += b

    return {
        k: v / max(count, 1)
        for k, v in totals.items()
    }

## Train A1

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

use_amp = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

best_val = float("inf")
bad_epochs = 0
history = []

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for epoch in range(1, EPOCHS + 1):
    model.train()

    running = {
        "total": 0.0,
        "mag": 0.0,
        "complex": 0.0,
        "phase": 0.0
    }
    seen = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS}"
    )

    for clean, noisy in pbar:
        clean = clean.to(
            device,
            non_blocking=True
        )
        noisy = noisy.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp
        ):
            clean_spec = stft_complex(clean)
            noisy_spec = stft_complex(noisy)

            outputs = model(noisy_spec)
            losses = phase_aware_loss(
                outputs,
                clean_spec
            )

        scaler.scale(
            losses["total"]
        ).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        scaler.step(optimizer)
        scaler.update()

        b = clean.size(0)

        for k in running:
            running[k] += (
                losses[k].item() * b
            )

        seen += b

        pbar.set_postfix(
            total=running["total"]/max(seen, 1),
            mag=running["mag"]/max(seen, 1),
            phase=running["phase"]/max(seen, 1)
        )

    train_stats = {
        k: v / max(seen, 1)
        for k, v in running.items()
    }

    val_stats = validate()

    row = {
        "epoch": epoch,
        "train_total": train_stats["total"],
        "train_mag": train_stats["mag"],
        "train_complex": train_stats["complex"],
        "train_phase": train_stats["phase"],
        "val_total": val_stats["total"],
        "val_mag": val_stats["mag"],
        "val_complex": val_stats["complex"],
        "val_phase": val_stats["phase"],
    }

    history.append(row)

    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "history.csv",
        index=False
    )

    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_total": val_stats["total"],
        "seed": SEED,
        "settings": {
            "sr": SR,
            "n_fft": N_FFT,
            "win_length": WIN_LENGTH,
            "hop_length": HOP_LENGTH,
            "segment_seconds": SEGMENT_SECONDS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "lambda_complex": LAMBDA_COMPLEX,
            "lambda_phase": LAMBDA_PHASE,
            "base_channels": 16,
            "depth": 4
        }
    }

    torch.save(
        state,
        OUTPUT_DIR / "last.pt"
    )

    if val_stats["total"] < best_val:
        best_val = val_stats["total"]
        bad_epochs = 0

        torch.save(
            state,
            OUTPUT_DIR / "best.pt"
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch}: "
        f"train_total={train_stats['total']:.6f}, "
        f"val_total={val_stats['total']:.6f}, "
        f"val_mag={val_stats['mag']:.6f}, "
        f"val_phase={val_stats['phase']:.6f}, "
        f"best={best_val:.6f}"
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

if torch.cuda.is_available():
    print(
        "Peak GPU memory allocated (MB):",
        torch.cuda.max_memory_allocated() / (1024**2)
    )

print("Training complete.")
print("Best checkpoint:", OUTPUT_DIR / "best.pt")

## Plot training curves

In [ ]:
import matplotlib.pyplot as plt

hist = pd.read_csv(
    OUTPUT_DIR / "history.csv"
)

plt.figure(figsize=(8, 5))
plt.plot(
    hist["epoch"],
    hist["train_total"],
    label="Train total"
)
plt.plot(
    hist["epoch"],
    hist["val_total"],
    label="Validation total"
)
plt.xlabel("Epoch")
plt.ylabel("Total loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    hist["epoch"],
    hist["val_mag"],
    label="Validation magnitude"
)
plt.plot(
    hist["epoch"],
    hist["val_phase"],
    label="Validation phase"
)
plt.xlabel("Epoch")
plt.ylabel("Loss component")
plt.legend()
plt.grid(alpha=0.2)
plt.show()